# Using lifespan models to make predictions on new data ("Transfer")

Created by Saige rutherford adapted by Charlotte Fraza and Johanna Bayer

In this tutorial, we demonstrate how to use large, pretrained normative models to generate predictions for new datasets.  We call this operation a **model transfer**. By transferring, hence applying, the parameters of these models to new data, you can take advantage of the statistical power of large reference cohorts without performing computationally expensive training.

In this script we will demonstrate how to:

1. Download and unzip the pretrained normative models
2. Load and prepare the transfer datasets (in this example case from OpenNeuro)
3. Configure covariates and response variables for the transfer function
4. Adapt the model to a new dataset
5. Run the transfer prediction using the PCNtoolkit

First, we install and import the required libraries. These include some libraries to download the pre-trained models, the pcntookit and some libraries to manipulate data in Python.

In [1]:
# Install the requests library if it's not already installed
try:
    import requests
except ModuleNotFoundError:
    %pip install requests

In [ ]:
import os
import requests
import xml.etree.ElementTree as ET
from urllib.parse import unquote
import zipfile
import pcntoolkit as ptk
import pandas as pd
import numpy as np

# Define the WebDAV URL and authentication credentials for Surfdrive to pull the model files from
BASE_URL = "https://surfdrive.surf.nl/public.php/webdav/zip/"
SHARE_TOKEN = "Mb6mZyFmJeCaPcZ"
PASSWORD = ""


### 1. Download and unzip the pretrained normative models

In the next step, need to get the models into your computing environment. We are going to download all the models from surfdrive: https://surfdrive.surf.nl/s/Mb6mZyFmJeCaPcZ?dir=/zip and store them in a new folder that we create in your local environment.

We first read out the names of all model folders in the remote directory:

In [3]:

# Define the working directory where all ZIP files will be downloaded
data_dir = os.path.abspath("../braincharts/data/")
os.makedirs(data_dir, exist_ok=True)

# Send a PROPFIND request to list the contents of the remote directory
headers = {"Depth": "1"}
r = requests.request("PROPFIND", BASE_URL, auth=(SHARE_TOKEN, PASSWORD), headers=headers)

# Parse the XML response returned by the WebDAV server
tree = ET.fromstring(r.content)

# Extract the names of all ZIP files in the remote directory
zip_files = []
for elem in tree.iter():
    if elem.tag.endswith("href"):
        name = unquote(elem.text.split("/")[-1])
        if name.lower().endswith(".zip"):
            zip_files.append(name)


print("Available models:")
print('\n'.join(zip_files))

Available models:
BLRw_ct_DES_lifespan_67K_89sites.zip
BLRw_fa_JHU_lifespan_24K_19sites.zip
BLRw_fc_yeo17_lifespan_21K_40sites.zip
BLRw_sa_DES_lifespan_37K_66sites.zip
BLRw_sa_DK_lifespan_46K_59sites.zip
BLRw_sc_lifespan_67K_89sites.zip
HBR_Sb_ct_DES_lifespan_79K_100sites.zip
HBR_Sb_ct_DK_lifespan_79K_100sites.zip
HBR_Sb_sa_DES_lifespan_37K_66sites.zip
HBR_Sb_sa_DK_lifespan_46K_59sites.zip
HBR_Sb_sc_lifespan_79K_100sites.zip


Now that you can see the available model names above, fill in your configuration here.
**This is the only cell you need to edit** before running the rest of the notebook.

- `MODEL_TO_DOWNLOAD` — which model to download:
  - a model name string (without `.zip`) — download that model only
  - `'all'` — download every available model (warning: several GB total)
  - `None` — skip download (model already present on disk)
- `MODEL_TO_TRANSFER` — the model to use for the transfer (must be a specific name string)
- `TARGET_DATA` — path to the CSV file with the data you want to transfer the model to


In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────

# Which model to download (see available names printed above):
#   'ModelName'  ->  download that model only
#   'all'        ->  download all models (warning: several GB)
#   None         ->  skip download (model already on disk)
MODEL_TO_DOWNLOAD = 'BLRw_ct_DES_lifespan_67K_89sites'

# Which model to use for the transfer (must be a specific name string):
MODEL_TO_TRANSFER = 'BLRw_ct_DES_lifespan_67K_89sites'

# Path to the dataset to transfer the model to:
TARGET_DATA = '../data/openneuro_lifespan_txfr.csv'

# ──────────────────────────────────────────────────────────────────────────────


Then we iterate over the model folder names and download each of them onto your local machine. Execution of this step may take long to run due to the large size (GBs) of the HBR models.

In [ ]:
# Determine which files to download based on MODEL_TO_DOWNLOAD
if MODEL_TO_DOWNLOAD is None:
    print('MODEL_TO_DOWNLOAD is None - skipping download.')
    zip_files_to_download = []
elif MODEL_TO_DOWNLOAD == 'all':
    zip_files_to_download = zip_files  # all files listed above
else:
    zip_files_to_download = [MODEL_TO_DOWNLOAD + '.zip']

# Download each ZIP file; skip files that are already present on disk
for fname in zip_files_to_download:
    out_path = os.path.join(data_dir, fname)

    if os.path.exists(out_path):
        print(f'{fname} already downloaded, skipping.')
        continue

    url = BASE_URL + fname
    resp = requests.get(url, auth=(SHARE_TOKEN, PASSWORD), stream=True)

    if resp.status_code != 200:
        print(f'Failed to download {fname} (HTTP {resp.status_code}), skipping.')
        continue

    print(f'Downloading {fname} ...')
    with open(out_path, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
    print(f'{fname} done.')


You can now inspect the zipped model files. Your folder should look something like this:

`BLRw_ct_DES_lifespan_67K_89sites.zip` <br>
`HBR_Sb_ct_DES_lifespan_79K_100sites.zip` <br>
`BLRw_fa_JHU_lifespan_24K_19sites.zip` <br>
`HBR_Sb_ct_DK_lifespan_79K_100sites.zip` <br>
`BLRw_sc_lifespan_67K_89sites.zip` <br>
`HBR_Sb_sc_lifespan_79K_100sites.zip` <br>
`BLRw_sa_DES_lifespan_37K_66sites.zip` <br>
`BLRw_sa_DK_lifespan_40K_59sites.zip` <br>
`BLRw_fc_yeo17_lifespan_21K_40sites.zip` <br>
`HBR_Sb_sa_DES_lifespan_37K_66sites.zip` <br>
`HBR_Sb_DK_lifespan_40K_59sites.zip`

Let’s break down what these filenames mean:
- BLRw: Bayesian Linear Regression model (Fraza et al.)
- HBR_Sb: Hierarchical Bayesian Regression model using a SHASH likelihood (de Boer et al., 2024)
- ct: cortical thickness
- fa: fractional anisotropy
- sc: subcortical measures
- DES: Destrieux atlas parcellation
- DK: Desikan–Killiany atlas parcellation
- 67K / 79K / 24K: number of subjects used to train the model
- 89sites / 100sites / 19sites: number of sites in the training dataset

Then we extract the models:

In [ ]:

# Define paths to the zipped model file and the directory where it will be extracted
if MODEL_TO_TRANSFER is None or MODEL_TO_TRANSFER == 'all':
    raise ValueError(
        "MODEL_TO_TRANSFER must be a specific model name string "
        "(e.g. MODEL_TO_TRANSFER = 'BLRw_ct_DES_lifespan_67K_89sites'). "
        "Available model names were printed above."
    )

zip_path = os.path.join(data_dir, MODEL_TO_TRANSFER + '.zip')
model_dir = os.path.join(data_dir, MODEL_TO_TRANSFER)

# Unzip the pretrained model if it has not already been extracted
if not os.path.exists(model_dir):
    os.makedirs(model_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(model_dir)


### 2. Load and prepare the transfer datasets (in this example case from OpenNeuro)

This section focuses on preparing the new data for which we want to generate predictions.

In general, the data should be split into two folds:
- The adaptation dataset (_ad_), which is used to estimate site differences between the new data and the pretrained models. This dataset is used only for adaptation and is not included in the final analysis.
- The test dataset (_te_), for which predictions will be generated.

Predictions are made only for the test set.

For this example, we use a publicly available dataset from OpenNeuro, composed of the following collection IDs: ds003416, ds003469, ds003826, ds000222 and ds000115. For convenience, this is provided as part of this repository.

If you plan to use your own data, this is the section you will most likely need to adapt. Your data needs to be split into two folds in a stratified manner, such that the distributions of relevant covariates (e.g., age, sex, site) are represented in both folds. You can use an in-built function for that, or you can create your own custom split. You will also have to make sure that the sex encoding corresponds with the pretrained models (F/M in the BLR case, 0/1 in the HBR SHASH case, with 0 representing female.)

First, we create an output directory.

In [5]:
# Create an output directory for transfer predictions
out_dir = model_dir + "_txfr"
os.makedirs(out_dir, exist_ok=True)

Next, we load the necessary information from the pretrained model

In [6]:
# Load the pretrained normative model
model = ptk.NormativeModel.load(model_dir)

# Extract model configuration: batch effects (e.g. site, sex), covariates (e.g. age),
# and response variables (brain measures predicted by the model)
batch_effects = list(model.unique_batch_effects.keys())
covariates = model.covariates
response_vars = model.response_vars

Next, we configure covariates and response variables for the transfer function.

In [7]:
df = pd.read_csv(TARGET_DATA, index_col=0)

# Recode sex variable to match the format expected by the model (F/M instead of 0/1)
df["sex"] = df["sex"].map({0: "F", 1: "M"})

# Create a subject ID column from the dataframe index
df["sub_id"] = df.index.astype(str)

# Keep only those response variables that are present in the dataframe
# (prevents errors when the model expects variables that are not in the CSV files)
response_vars = [v for v in response_vars if v in df.columns]


Next, we configure a NormData object for the transfer data and split the data into a training and test set. 

In the transfer case, the training data is used to learn the batch effects for the unseen data and the test set is used to evaluate the model.

In [9]:
# ensure all to use just fitted models
fitted_response_vars = [
    rv for rv in response_vars
    if rv in model.response_vars and model[rv].is_fitted
]

skipped_response_vars = [
    rv for rv in response_vars
    if rv not in fitted_response_vars
]
print(f"Using {len(fitted_response_vars)} fitted response variables")
print("Skipped (not fitted):", skipped_response_vars)

# Create a NormData object to prepare the data for normative modelling
norm_data = ptk.NormData.from_dataframe(
    name="pu25_thickness_txfr",
    dataframe=df,
    batch_effects=batch_effects,
    subject_ids="sub_id",
    response_vars=fitted_response_vars,
    covariates=covariates,
    remove_Nan=True,
    remove_outliers=True,
    z_threshold=10
)

## In case you want to use your own train-test split instead of an automatically generated one, use these lines instead
#train = ptk.NormData.from_dataframe(name="ad", dataframe=df_ad, covariates=covariates, batch_effects=batch_effects, response_vars=response_vars,
# remove_Nan=True,remove_outliers=True,z_threshold=10)
#test = ptk.NormData.from_dataframe(name="te", dataframe=df_te, covariates=covariates, batch_effects=batch_effects, response_vars=response_vars,
# remove_Nan=True,remove_outliers=True,z_threshold=10))
  
# Split the data into adaptation (train) and test sets
train, test = norm_data.train_test_split(splits=[0.5, 0.5])


Using 149 fitted response variables
Skipped (not fitted): ['lh_G&S_frontomargin_thickness']
Process: 9708 - 2026-05-07 07:55:55 - Removed 0 NANs
Process: 9708 - 2026-05-07 07:55:55 - Removed 0 outliers
Process: 9708 - 2026-05-07 07:55:56 - Dataset "pu25_thickness_txfr" created.
    - 459 observations
    - 431 unique subjects
    - 1 covariates
    - 149 response variables
    - 2 batch effects:
    	site (5)
	sex (2)
    


### 3. Run the transfer prediction using PCNtoolkit routines

Now, we are ready to transfer the model

In [10]:
# Run transfer prediction to adapt the pretrained model and compute deviation scores
transferred_model = model.transfer_predict(train, test, save_dir=out_dir)

Process: 9708 - 2026-05-07 07:56:01 - Transferring models on 149 response variables.
Process: 9708 - 2026-05-07 07:56:01 - Transferring model for rh_S_circular_insula_sup_thickness.
Process: 9708 - 2026-05-07 07:56:01 - Transferring model for lh_S_parieto_occipital_thickness.
Process: 9708 - 2026-05-07 07:56:01 - Transferring model for rh_S_postcentral_thickness.
Process: 9708 - 2026-05-07 07:56:01 - Transferring model for lh_G_rectus_thickness.
Process: 9708 - 2026-05-07 07:56:01 - Transferring model for lh_Pole_occipital_thickness.
Process: 9708 - 2026-05-07 07:56:01 - Transferring model for lh_S_oc_sup&transversal_thickness.
Process: 9708 - 2026-05-07 07:56:01 - Transferring model for lh_S_precentral-inf-part_thickness.
Process: 9708 - 2026-05-07 07:56:01 - Transferring model for rh_G_temporal_middle_thickness.
Process: 9708 - 2026-05-07 07:56:01 - Transferring model for lh_S_circular_insula_sup_thickness.
Process: 9708 - 2026-05-07 07:56:01 - Transferring model for lh_G&S_subcentra

/Users/andmar/sfw/anaconda3/envs/py312dev/lib/python3.12/site-packages/pcntoolkit/util/evaluator.py:275: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, p_rho = stats.spearmanr(y, yhat)
/Users/andmar/sfw/anaconda3/envs/py312dev/lib/python3.12/site-packages/pcntoolkit/util/evaluator.py:370: RuntimeWarning: divide by zero encountered in log
  null_logp = -0.5 * np.log(2 * np.pi * sigma_null**2) - ((y - mu_null)**2) / (2 * sigma_null**2)
/Users/andmar/sfw/anaconda3/envs/py312dev/lib/python3.12/site-packages/pcntoolkit/util/evaluator.py:370: RuntimeWarning: invalid value encountered in divide
  null_logp = -0.5 * np.log(2 * np.pi * sigma_null**2) - ((y - mu_null)**2) / (2 * sigma_null**2)
/Users/andmar/sfw/anaconda3/envs/py312dev/lib/python3.12/site-packages/pcntoolkit/dataio/norm_data.py:1094: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for

Process: 9708 - 2026-05-07 08:00:36 - Dataset "centile" created.
    - 150 observations
    - 150 unique subjects
    - 1 covariates
    - 149 response variables
    - 2 batch effects:
    	site (1)
	sex (1)
    
Process: 9708 - 2026-05-07 08:00:36 - Computing centiles for 149 response variables.
Process: 9708 - 2026-05-07 08:00:36 - Computing centiles for rh_S_circular_insula_sup_thickness.
Process: 9708 - 2026-05-07 08:00:36 - Computing centiles for lh_S_parieto_occipital_thickness.
Process: 9708 - 2026-05-07 08:00:36 - Computing centiles for rh_S_postcentral_thickness.
Process: 9708 - 2026-05-07 08:00:36 - Computing centiles for lh_G_rectus_thickness.
Process: 9708 - 2026-05-07 08:00:36 - Computing centiles for lh_Pole_occipital_thickness.
Process: 9708 - 2026-05-07 08:00:36 - Computing centiles for lh_S_oc_sup&transversal_thickness.
Process: 9708 - 2026-05-07 08:00:36 - Computing centiles for lh_S_precentral-inf-part_thickness.
Process: 9708 - 2026-05-07 08:00:36 - Computing centil

/Users/andmar/sfw/anaconda3/envs/py312dev/lib/python3.12/site-packages/pcntoolkit/util/evaluator.py:275: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, p_rho = stats.spearmanr(y, yhat)
/Users/andmar/sfw/anaconda3/envs/py312dev/lib/python3.12/site-packages/pcntoolkit/util/evaluator.py:370: RuntimeWarning: divide by zero encountered in log
  null_logp = -0.5 * np.log(2 * np.pi * sigma_null**2) - ((y - mu_null)**2) / (2 * sigma_null**2)
/Users/andmar/sfw/anaconda3/envs/py312dev/lib/python3.12/site-packages/pcntoolkit/util/evaluator.py:370: RuntimeWarning: invalid value encountered in divide
  null_logp = -0.5 * np.log(2 * np.pi * sigma_null**2) - ((y - mu_null)**2) / (2 * sigma_null**2)
/Users/andmar/sfw/anaconda3/envs/py312dev/lib/python3.12/site-packages/pcntoolkit/dataio/norm_data.py:1094: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for

Process: 9708 - 2026-05-07 08:05:58 - Dataset "centile" created.
    - 150 observations
    - 150 unique subjects
    - 1 covariates
    - 149 response variables
    - 2 batch effects:
    	site (1)
	sex (1)
    
Process: 9708 - 2026-05-07 08:05:58 - Computing centiles for 149 response variables.
Process: 9708 - 2026-05-07 08:05:58 - Computing centiles for rh_S_circular_insula_sup_thickness.
Process: 9708 - 2026-05-07 08:05:58 - Computing centiles for lh_S_parieto_occipital_thickness.
Process: 9708 - 2026-05-07 08:05:58 - Computing centiles for rh_S_postcentral_thickness.
Process: 9708 - 2026-05-07 08:05:58 - Computing centiles for lh_G_rectus_thickness.
Process: 9708 - 2026-05-07 08:05:59 - Computing centiles for lh_Pole_occipital_thickness.
Process: 9708 - 2026-05-07 08:05:59 - Computing centiles for lh_S_oc_sup&transversal_thickness.
Process: 9708 - 2026-05-07 08:05:59 - Computing centiles for lh_S_precentral-inf-part_thickness.
Process: 9708 - 2026-05-07 08:05:59 - Computing centil